In [13]:
import pandas as pd
import re
import emoji
from collections import Counter
import numpy as np
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import os
from dotenv import load_dotenv
from supabase import create_client, Client

In [14]:
load_dotenv()

URL = os.getenv("SUPABASE_URL")
KEY = os.getenv("SUPABASE_KEY")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")

# Validasi cepat agar skrip tidak diam-diam gagal jika .env lupa dibuat
if not all([URL, KEY, BUCKET_NAME]):
    raise ValueError("Kredensial Supabase tidak lengkap. Cek file .env!")

In [15]:
# ==========================================
# CELL 2: LOAD DATA & DICTIONARIES
# ==========================================
print("Loading data and dictionaries...")
df = pd.read_csv(f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_clean.csv")

print(f"Sukses! Menarik {len(df)} baris langsung dari awan.")
display(df.head())

# 2. Load Kamus Alay
df_alay = pd.read_csv('./Resource/colloquial-indonesian-lexicon.csv')
slang_dict = dict(zip(df_alay['slang'], df_alay['formal']))

# 3. Emoji Dictionary (Teroptimasi dari hasil ekstraksi dataset)
emoji_dict = {
    '😡': ' marah ', '😠': ' marah ', '🤬': ' marah ',
    '😭': ' sedih ', '😢': ' sedih ', '😞': ' kecewa ',
    '👍': ' bagus ', '😍': ' suka ', '🥰': ' suka ', '💯': ' sempurna ',
    '👎': ' jelek ', '💩': ' sampah ', '🙏': ' tolong ',
    '😅': ' bingung ', '😂': ' tertawa ', '🤣': ' tertawa ', 
    '🤦': ' kecewa ', '😁': ' senang ', '⭐': ' bintang ', 
    '🤩': ' kagum ', '🌟': ' bintang ', '😇': ' baik ',    
    '😲': ' kaget ', '♂': ' '
}

Loading data and dictionaries...
Sukses! Menarik 34929 baris langsung dari awan.


,review_text,label,date,app_version
0,sangat bagus,1,2026-01-06 18:59:47,2.0.4
1,"ambil foto diri error, loading lama dan lemot",0,2026-01-06 15:33:05,2.0.4
2,aplikasi bikin stressss...mau masuk ribet lg.....,0,2026-01-06 14:51:18,2.0.4
3,"pelayannya g ada, saya udah coba ajukan ulang ...",1,2026-01-06 14:40:48,2.0.4
4,maksud tujuan nya bagus. tapi kebanyakan aplik...,0,2026-01-06 13:30:05,2.0.4


In [16]:
# ==========================================
# CELL 3: STOPWORD CONFIGURATION
# ==========================================
factory = StopWordRemoverFactory()
standard_stopwords = factory.get_stop_words()

# Kata penting yang diselamatkan untuk Dataset C (Ablation Study)
important_words = ['tidak', 'bukan', 'jangan', 'belum', 'kurang', 'sangat', 'paling', 'agak', 'terus']
custom_stopwords = [word for word in standard_stopwords if word not in important_words]

# Konversi ke Set untuk pencarian O(1) yang super cepat
standard_stopwords = set(standard_stopwords)
custom_stopwords = set(custom_stopwords)

In [17]:
# ==========================================
# CELL 4: CORE PREPROCESSING FUNCTIONS
# ==========================================
def clean_and_normalize(text):
    text = str(text).lower()
    
    # A. Terjemahkan Emoji SEBELUM tanda baca dihapus
    for emo, word in emoji_dict.items():
        text = text.replace(emo, word)
        
    # B. Hapus URL, Mention, dan Angka
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\d+', '', text)
    
    # C. Hapus tanda baca (sisakan huruf dan spasi)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # D. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # E. Normalisasi Kata Gaul (Slang)
    words = text.split()
    normalized_words = [slang_dict.get(w, w) for w in words]
    
    return ' '.join(normalized_words)

def remove_standard_stopwords(text):
    return ' '.join([w for w in text.split() if w not in standard_stopwords])

def remove_custom_stopwords(text):
    return ' '.join([w for w in text.split() if w not in custom_stopwords])

In [18]:
# 6. Fungsi Hapus Stopwords
def remove_standard_stopwords(text):
    return ' '.join([w for w in text.split() if w not in standard_stopwords])

def remove_custom_stopwords(text):
    return ' '.join([w for w in text.split() if w not in custom_stopwords])

In [19]:
# ==========================================
# CELL 5: EXECUTE PIPELINE
# ==========================================
print("Memulai proses pembersihan 34.000+ baris...")

# Dataset A: Pembersihan Dasar + Emoji + Slang (Tanpa Hapus Stopword)
df['text_baseline'] = df['review_text'].apply(clean_and_normalize)

# Dataset B: Menghapus Stopword Standard
df['text_standard'] = df['text_baseline'].apply(remove_standard_stopwords)

# Dataset C: Menghapus Stopword Custom (Cerdas)
df['text_custom'] = df['text_baseline'].apply(remove_custom_stopwords)

print("Proses string teks selesai.")

Memulai proses pembersihan 34.000+ baris...
Proses string teks selesai.


In [20]:
# ==========================================
# CELL 6: SMART MISSING VALUE HANDLING (PENTING!)
# ==========================================
# 1. Konversi string kosong menjadi NaN agar bisa dideteksi Pandas
df['text_baseline'] = df['text_baseline'].replace(r'^\s*$', np.nan, regex=True)
df['text_standard'] = df['text_standard'].replace(r'^\s*$', np.nan, regex=True)
df['text_custom'] = df['text_custom'].replace(r'^\s*$', np.nan, regex=True)

print(f'{len(df)} baris awal sebelum pembersihan final.')

# 2. HANYA drop baris yang text_baseline-nya benar-benar kosong (noise murni, ~27 baris)
df = df.dropna(subset=['text_baseline']).reset_index(drop=True)

# 3. PENYELAMATAN DATA: Kembalikan NaN menjadi string kosong khusus untuk kolom ablasi
# Ini memastikan ulasan yang isinya hanya kata hubung tetap dievaluasi oleh SVM
df['text_standard'] = df['text_standard'].fillna("")
df['text_custom'] = df['text_custom'].fillna("")

print(f"Pembersihan final selesai. Total data siap modeling: {len(df)} baris.")

34929 baris awal sebelum pembersihan final.
Pembersihan final selesai. Total data siap modeling: 34902 baris.


In [22]:
# ==========================================
# CELL 7: SAVE TO CSV
# ==========================================
# 2. Inisialisasi Klien Supabase
supabase: Client = create_client(URL, KEY)

# 3. Konfigurasi Jalur File
local_file_path = './Result/ulasan_ikd_preprocessed.csv'
supabase_file_name = 'ulasan_ikd_preprocessed.csv'

# 4. Automasi Upload (Pipeline Ekspor)
print(f"Mengunggah {local_file_path} ke Supabase Cloud Storage...")

try:
    with open(local_file_path, 'rb') as f:
        # Mengunggah file dengan opsi 'upsert' agar otomatis menimpa file lama
        response = supabase.storage.from_(BUCKET_NAME).upload(
            file=f,
            path=supabase_file_name,
            file_options={"content-type": "text/csv", "x-upsert": "true"}
        )
    print("Upload berhasil! Data di cloud sudah diperbarui.")
except Exception as e:
    print(f"Terjadi kesalahan saat upload: {e}")

print("✅ MLOps Pipeline Sempurna! Dataset versi terbaru sudah online di Supabase.")

Mengunggah ./Result/ulasan_ikd_preprocessed.csv ke Supabase Cloud Storage...
Upload berhasil! Data di cloud sudah diperbarui.
✅ MLOps Pipeline Sempurna! Dataset versi terbaru sudah online di Supabase.
